# Home Visitation Follow-Up Prioritization

## 1) Problem Framing
Prioritize visits with `follow_up_needed` that are likely to lead to adverse follow-on outcomes.

**Target:** adverse `visit_outcome` or subsequent incident within 60 days (proxy from merged data).


In [ ]:
import warnings
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, mean_squared_error, r2_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, mean_absolute_error, average_precision_score,
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor,
)
SEED = 42
pd.set_option("display.max_columns", 200)
DATA_DIR = Path("../lighthouse_csv_v7/lighthouse_csv_v7")
assert DATA_DIR.exists(), f"Missing data: {DATA_DIR.resolve()}"

visits = pd.read_csv(DATA_DIR / "home_visitations.csv", parse_dates=["visit_date"]).sort_values("visit_date")
visits["follow_num"] = visits["follow_up_needed"].astype(str).str.lower().isin(["true", "1", "yes"]).astype(int)
# Simple proxy target: safety concern or poor cooperation -> adverse
v = visits.copy()
bad = (
    v["safety_concerns_noted"].astype(str).str.lower().isin(["true", "1", "yes"])
    | (v["family_cooperation_level"].astype(str).str.lower().isin(["low", "poor", "1", "2"]))
)
v["adverse_proxy"] = bad.astype(int)
res = pd.read_csv(DATA_DIR / "residents.csv")
v = v.merge(res[["resident_id", "present_age", "initial_risk_level", "current_risk_level", "safehouse_id"]], on="resident_id", how="left")
X = v[["visit_type", "family_cooperation_level", "follow_num", "present_age", "initial_risk_level", "current_risk_level", "safehouse_id"]].copy()
y = v["adverse_proxy"].astype(int)
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
prep = ColumnTransformer(
    [
        ("num", Pipeline([("im", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("im", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y if y.nunique() > 1 else None)
exp_m = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))])
pred_m = Pipeline([("prep", prep), ("clf", GradientBoostingClassifier(random_state=SEED))])
exp_m.fit(X_tr, y_tr)
pred_m.fit(X_tr, y_tr)
proba = pred_m.predict_proba(X_te)[:, 1]
print("ROC-AUC:", roc_auc_score(y_te, proba) if y_te.nunique() > 1 else "n/a")
print(classification_report(y_te, (proba >= 0.5).astype(int), digits=3))


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pipeline_dashboard_output import export_generic_classifier_dashboard, save_dashboard_json
_fn = exp_m.named_steps["prep"].get_feature_names_out()
_coef = pd.DataFrame({"feature": _fn, "coef": exp_m.named_steps["clf"].coef_[0]})
_coef["abs_coef"] = _coef["coef"].abs()
_imp = pd.DataFrame({"feature": _fn, "importance": pred_m.named_steps["clf"].feature_importances_})
_dash = export_generic_classifier_dashboard(
    pipeline_id="home-visitation-followup-prioritization",
    display_name="Home visitation follow-up prioritization",
    problem_summary="Prioritizes visits where adverse follow-on risk (safety/cooperation proxy) is elevated.",
    pred_m=pred_m,
    exp_m=exp_m,
    X_te=X_te,
    y_te=y_te,
    proba=proba,
    coef_df=_coef,
    imp_df=_imp,
    positive_class_description="adverse visit-level proxy outcome",
)
save_dashboard_json(_dash)
print("dashboard_export:", _dash.get("export_path"))


## Note
Refine target by joining subsequent `incident_reports` on resident and date window in production.

## 6) Deployment
Field triage queue sorted by risk score.
